In [1]:
# ############################################################
# 이상치탐지 예시 — IsolationForest로 '튀는 값' 찾기 (-1/1)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기
# ------------------------------------------------------------
import pandas as pd                               # 표 다루기 (엑셀 같은 표를 코드로 다루기)
from sklearn.datasets import load_breast_cancer   # 예시용 데이터 (연습에 자주 쓰는 유방암 데이터)
from sklearn.preprocessing import StandardScaler  # 스케일링 (숫자 크기를 공평하게 맞추기)
from sklearn.ensemble import IsolationForest      # 이상치 탐지 (빨리 고립될수록 이상치로 보는 방법)

In [4]:
# ------------------------------------------------------------
# [데이터 살펴보기 · EDA] 유방암 진단 데이터는 어떻게 생겼나
#   · 세포핵 측정치 30개(반지름·질감·둘레·면적 등)로 종양을 판단하는 데이터
#   · 행 = 환자 샘플, 열 = 특성 30개 + target(0=악성, 1=양성)
#   · 이번엔 정답(target)은 쓰지 않고 '유별나게 튀는 샘플'만 찾는다(비지도)
# ------------------------------------------------------------
df = load_breast_cancer(as_frame=True).frame      # 표로 불러오기 (행=환자, 열=측정치인 표로 받기)
print('데이터 크기(행, 열):', df.shape)           # 크기 확인 (샘플 569명 × 열 31개인지 보기)
df.head()                                          # 앞 5줄 미리보기 (어떤 값들이 들었는지 눈으로)

# df['target'].unique()

데이터 크기(행, 열): (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
# ------------------------------------------------------------
# [데이터 살펴보기] 대표 특성 3개의 요약 통계 (값 범위 확인)
#   → 열마다 숫자 크기(범위)가 제각각인 게 보임 → 그래서 다음에 '스케일링'이 필요
# ------------------------------------------------------------
df[['mean radius', 'mean texture', 'mean area']].describe().round(2)   # 요약 통계 (평균·최소·최대를 한눈에)

,mean radius,mean texture,mean area
count,569.00,569.00,569.00
mean,14.13,19.29,654.89
std,3.52,4.30,351.91
min,6.98,9.71,143.50
25%,11.70,16.17,420.30
50%,13.37,18.84,551.10
75%,15.78,21.80,782.70
max,28.11,39.28,2501.00


In [6]:
# ------------------------------------------------------------
# [목적] 데이터 준비 — 정답(target)은 빼고 특성 30개만 사용 (비지도)
# ------------------------------------------------------------
X = df.drop(columns='target')                     # 특성만 남기기 (정답인 target 열을 빼기)
print('사용할 특성 개수:', X.shape[1])            # 특성 수 확인 (30개인지 보기)

사용할 특성 개수: 30


In [8]:
# ------------------------------------------------------------
# [목적] 스케일링 후 학습+판정 한 번에 (contamination = 이상치 비율 힌트)
#   (위 describe에서 봤듯 열마다 값 범위가 달라 스케일링이 필수)
# ------------------------------------------------------------
X_s = StandardScaler().fit_transform(X)           # 스케일링 (모든 특성을 같은 잣대로 맞추기)
iso = IsolationForest(contamination=0.05, random_state=0)  # 모델 만들기 (이상치 약 5%라고 귀띔, 결과 고정)
pred = iso.fit_predict(X_s)                        # 학습+판정 (각 샘플을 -1=이상 / 1=정상으로)
pred[:10]

array([-1,  1,  1, -1,  1,  1,  1,  1,  1, -1])

In [6]:
# ------------------------------------------------------------
# [목적] 결과 읽기 — -1(이상)/1(정상)
# ------------------------------------------------------------
print('전체:', len(X_s), '| 이상치(-1):', (pred == -1).sum(), '| 정상(1):', (pred == 1).sum())  # 개수 세기 (전체·이상·정상)

전체: 569 | 이상치(-1): 29 | 정상(1): 540


In [9]:
# ------------------------------------------------------------
# [목적] 예측 + 이상치 점수(anomaly score)를 표에 열로 추가
#   · predict           -> 1(정상) / -1(이상)
#   · decision_function -> 이상치 점수 (낮을수록·0보다 작을수록 이상치 가능성 큼)
# ------------------------------------------------------------
df['anomaly_prediction'] = iso.predict(X_s)        # 판정 결과를 새 열로 (1=정상 / -1=이상)
df['anomaly_score'] = iso.decision_function(X_s)   # 이상치 '점수'도 열로 (낮을수록 더 유별남)

df.sort_values('anomaly_score').head()             # 점수 낮은 순 = 가장 이상한 샘플부터 보기

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,anomaly_prediction,anomaly_score
122,24.250,20.20,166.20,1761.0,0.1447,0.2867,0.4268,0.20120,0.2655,0.06877,...,2073.0,0.1696,0.4244,0.5803,0.2248,0.3222,0.08009,0,-1,-0.145682
212,28.110,18.47,188.50,2499.0,0.1142,0.1516,0.3201,0.15950,0.1648,0.05525,...,2499.0,0.1142,0.1516,0.3201,0.1595,0.1648,0.05525,0,-1,-0.143636
461,27.420,26.27,186.90,2501.0,0.1084,0.1988,0.3635,0.16890,0.2061,0.05623,...,4254.0,0.1357,0.4256,0.6833,0.2625,0.2641,0.07427,0,-1,-0.143528
152,9.731,15.34,63.78,300.2,0.1072,0.1599,0.4108,0.07857,0.2548,0.09296,...,380.5,0.1292,0.2772,0.8216,0.1571,0.3108,0.12590,1,-1,-0.130035
108,22.270,19.67,152.80,1509.0,0.1326,0.2768,0.4264,0.18230,0.2556,0.07039,...,2360.0,0.1701,0.6997,0.9608,0.2910,0.4055,0.09789,0,-1,-0.108001


In [10]:
# ------------------------------------------------------------
# [목적] 이상치인 행만 따로 빼서 살펴보기 (DataFrame으로)
# ------------------------------------------------------------
outliers = df[df['anomaly_prediction'] == -1]      # 이상치(-1)인 행만 골라내기 (튀는 샘플만 모으기)
print('이상치 개수:', len(outliers))              # 몇 개인지 (29개)
outliers.sort_values('anomaly_score').head()       # 가장 이상한(점수 낮은) 순으로 표 확인

이상치 개수: 29


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,anomaly_prediction,anomaly_score
122,24.250,20.20,166.20,1761.0,0.1447,0.2867,0.4268,0.20120,0.2655,0.06877,...,2073.0,0.1696,0.4244,0.5803,0.2248,0.3222,0.08009,0,-1,-0.145682
212,28.110,18.47,188.50,2499.0,0.1142,0.1516,0.3201,0.15950,0.1648,0.05525,...,2499.0,0.1142,0.1516,0.3201,0.1595,0.1648,0.05525,0,-1,-0.143636
461,27.420,26.27,186.90,2501.0,0.1084,0.1988,0.3635,0.16890,0.2061,0.05623,...,4254.0,0.1357,0.4256,0.6833,0.2625,0.2641,0.07427,0,-1,-0.143528
152,9.731,15.34,63.78,300.2,0.1072,0.1599,0.4108,0.07857,0.2548,0.09296,...,380.5,0.1292,0.2772,0.8216,0.1571,0.3108,0.12590,1,-1,-0.130035
108,22.270,19.67,152.80,1509.0,0.1326,0.2768,0.4264,0.18230,0.2556,0.07039,...,2360.0,0.1701,0.6997,0.9608,0.2910,0.4055,0.09789,0,-1,-0.108001


In [11]:
# ------------------------------------------------------------
# [목적] 이상치 제거 — 정상(1)만 남긴 '깨끗한' 데이터 만들기
# ------------------------------------------------------------
df_clean = df[df['anomaly_prediction'] == 1].drop(columns=['anomaly_prediction', 'anomaly_score'])  # 정상만 남기고 보조열 제거
print('원본:', len(df), '행  ->  정제 후:', len(df_clean), '행')   # 개수 비교 (569 -> 540)
print('제거된 이상치:', len(df) - len(df_clean), '개')            # 몇 개 뺐나 (29)
df_clean.head()                                    # 정제된 표 미리보기 (튀는 값이 빠진 상태)

원본: 569 행  ->  정제 후: 540 행
제거된 이상치: 29 개


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.2,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.1578,0.08089,0.2087,0.07613,...,23.75,103.4,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.12440,0
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.1127,0.07400,0.1794,0.05742,...,27.66,153.2,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0


In [10]:
# ============================================================
# [결과 해석]
#  · 569개 중 29개(약 5%)를 이상치(-1)로, 나머지 540개는 정상(1)으로 판정
#  · anomaly_score(점수)로 '얼마나 이상한지' 순위까지 매길 수 있음(낮을수록 더 이상)
#  · pred==1 만 남기면 '튀는 값 제거' -> 데이터 정제(df_clean) 완료
#  · 주의: 판정 결과는 항상 -1(이상) / 1(정상) 두 종류뿐
# ============================================================

In [12]:
# ------------------------------------------------------------
# [실습] 이상치를 '제거하면' 이진분류가 더 잘될까? (정확도·ROC-AUC 비교)
#   · 유방암엔 정답 target(0=악성,1=양성)이 있어 '이진분류'로 검증 가능
#   · 같은 시험셋(test)으로, 훈련셋만 [원본] vs [이상치 제거]로 바꿔 비교
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

y = df['target']                                  # 정답 (0=악성, 1=양성)
Xf = df[list(load_breast_cancer().feature_names)] # 원래 특성 30개만 (보조열 제외)
X_tr, X_te, y_tr, y_te = train_test_split(
    Xf, y, test_size=0.2, random_state=0, stratify=y)   # 8:2로 나누기 (비율 유지)

def score(A, b):                                  # 학습 후 '같은 시험셋'으로 정확도·AUC 반환
    m = LogisticRegression(max_iter=5000).fit(A, b)
    return accuracy_score(y_te, m.predict(X_te)), roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

acc0, auc0 = score(X_tr, y_tr)                     # (1) 원본 — 이상치 그대로 학습

mask = IsolationForest(contamination=0.05, random_state=0).fit_predict(X_tr) == 1  # 훈련셋 정상만
acc1, auc1 = score(X_tr[mask], y_tr[mask])         # (2) 이상치 제거 후 학습

print(f'[원본]       정확도 {acc0:.3f} | ROC-AUC {auc0:.3f} | 훈련 {len(X_tr)}개')
print(f'[이상치 제거] 정확도 {acc1:.3f} | ROC-AUC {auc1:.3f} | 훈련 {int(mask.sum())}개')

[원본]       정확도 0.947 | ROC-AUC 0.992 | 훈련 455개
[이상치 제거] 정확도 0.947 | ROC-AUC 0.992 | 훈련 432개


In [12]:
# ============================================================
# [결과 해석] 이상치 제거가 '늘' 이득은 아니다
#  · 정확도 0.947 -> 0.947,  ROC-AUC 0.992 -> 0.992 로 거의 그대로
#  · 이 데이터는 이미 깨끗한 편이라 5%를 빼도 성능 변화가 미미함
#  · 교훈: 이상치를 무작정 지우지 말고 '성능(정확도·AUC)으로 검증'한 뒤 결정!
#      (이상치가 진짜 신호일 수도 있어 함부로 버리면 오히려 손해)
# ============================================================